# Notebook B: train and evaluate the LSTM

Trains a sequence-to-sequence LSTM that predicts the next 7 months of every index at a pixel from:
- the last 12 months of indices (as anomalies) and weather,
- the weather of the 7 forecast months (observed in training, SEAS5 at forecast time),
- static terrain features and the county.

**Reads** `training_dataset_clipped.npz` from Notebook A (indices, months and counties are read from the file).
**Writes** `model\final_model.pt` and skill tables to `F:\new_predict\VI_LSTM\model`, for Notebook C.

**Steps:** train on 2020-2023, validate on 2024, test on 2025 to Sep 2026, compare with climatology and persistence, then refit on all data.

**Reading the scores:** skill = 1 - MSE(model) / MSE(climatology). 0 = a normal month, 1 = perfect. Skill is highest at lead 1. The observed-weather test is an upper bound. Pixels are a stratified sample, so quote the per-county numbers.

In [ ]:
import os
# avoids OpenMP crash on Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import os, json, math, time, copy, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

WORK_DIR  = r"F:\new_predict\VI_LSTM"
DATA_NPZ  = os.path.join(WORK_DIR, "training_dataset_clipped.npz")      # falls back to training_dataset.npz
MODEL_DIR = os.path.join(WORK_DIR, "model"); os.makedirs(MODEL_DIR, exist_ok=True)
SEED = 42

# forecast geometry
L, H = 12, 7                         # months of history in, months ahead out
TRAIN_END, VAL_END = "2023-12", "2024-12"
HOLDOUT_COUNTY = None
PIX_PER_COUNTY = 30000

# data handling
PRIMARY        = "NDVI"
MIN_ENC_USABLE = 4
QA2_WEIGHT     = 0.3
QA_SOURCE      = "MCARI"
Z_CLIP         = 5.0

# climatology
K_SHRINK, ELEV_BINS, SMOOTH_CLIM = 2.0, 10, True

# model and training
HIDDEN, LAYERS, STATIC_EMB, DROPOUT = 64, 1, 8, 0.3
LR, WEIGHT_DECAY, BATCH = 5e-4, 1e-2, 512
EPOCH_SAMPLES, VAL_SAMPLES = 50_000, 60_000
MAX_EPOCHS, PATIENCE, N_SEEDS = 100, 10, 3
HUBER_DELTA, CLIP_NORM = 1.0, 1.0
WX_NOISE, WX_BIAS_SD = 0.3, 0.3

RUN_ROLLING_ORIGIN = False
ROLLING_SEEDS      = 1
FINAL_FIT          = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE)

## 1. Load the data

In [ ]:
path = DATA_NPZ
if not os.path.exists(path):
    path = DATA_NPZ.replace("_clipped", "")
    print("!! WARNING: training_dataset_clipped.npz not found, falling back to the UNCLIPPED file. Run the clipping cell at the end of Notebook A first:\n"
          "   Raw values would inflate the anomaly SD of the heavy-tailed indices.")
D = np.load(path, allow_pickle=False)
print("reading", path)
VIS      = [str(v) for v in D["vis"]]
COUNTIES = [str(c) for c in D["counties"]]
MONTHS   = [str(m) for m in D["months"]]
WXN      = [str(w) for w in D["wx_names"]]
STN      = [str(s) for s in D["static_names"]]
CID_all  = D["county_id"].astype(int)

keep = np.sort(np.concatenate([np.flatnonzero(CID_all == c)[:PIX_PER_COUNTY] for c in range(len(COUNTIES))]))
VI, QA, WX, ST, CID = D["vi"][keep], D["qa"][keep].copy(), D["wx"][keep], D["static"][keep], CID_all[keep]
N, M, K = VI.shape
_vs = {str(s): i for i, s in enumerate(D["vi_stat_names"])}; VST = D["vi_stats"]
for _k, _v in enumerate(VIS):
    _p01, _p99 = VST[_k, _vs["p01"]], VST[_k, _vs["p99"]]; _sp = _p99 - _p01
    _mx, _mn = np.nanmax(VI[:, :, _k]), np.nanmin(VI[:, :, _k])
    if _mx > _p99 + 2 * _sp or _mn < _p01 - 2 * _sp:
        print(f"!! WARNING: {_v} range {_mn:.3g} .. {_mx:.3g} is far outside its p01..p99 ({_p01:.3g} .. {_p99:.3g}): is this the clipped file?")
print(f"{N} pixels x {M} months ({MONTHS[0]} .. {MONTHS[-1]}) x {K} indices: {VIS}")

has_qa = (QA == 2).any(axis=(0, 1))
print("indices with their own QA band:", [v for v, h in zip(VIS, has_qa) if h])
if QA_SOURCE and QA_SOURCE in VIS and has_qa[VIS.index(QA_SOURCE)]:
    gf = QA[:, :, VIS.index(QA_SOURCE)] == 2
    for k, v in enumerate(VIS):
        if not has_qa[k]:
            QA[:, :, k] = np.where(gf & np.isfinite(VI[:, :, k]) & (QA[:, :, k] == 0), 2, QA[:, :, k])
    print(f"indices without a QA band now inherit the gap-filled flag of {QA_SOURCE}")

USABLE = np.isfinite(VI) & ((QA == 0) | (QA == 2))
CLEAN  = np.isfinite(VI) & (QA == 0)
MIDX = {m: i for i, m in enumerate(MONTHS)}
CAL  = np.array([int(m[5:7]) - 1 for m in MONTHS])
PRI_K = VIS.index(PRIMARY) if PRIMARY in VIS else 0
print("share of values: clear %.2f | gap-filled %.2f | missing %.2f" % (CLEAN.mean(), (USABLE & ~CLEAN).mean(), (~USABLE).mean()))
print("pixels per county:", {c: int((CID == i).sum()) for i, c in enumerate(COUNTIES)})
_ri = WXN.index("rh"); _yrs = np.array([int(m[:4]) for m in MONTHS])
for _ci, _c in enumerate(COUNTIES):
    _mm = np.nanmean(WX[CID == _ci][:, :, _ri], axis=0)                       # county-mean RH per month
    _full = [y for y in np.unique(_yrs) if (_yrs == y).sum() == 12]
    _tab = np.array([[_mm[(_yrs == y) & (CAL == c)][0] for c in range(12)] for y in _full])   # years x 12 calendar months
    _sp = np.nanstd(_tab, axis=0)
    if np.nanmedian(_sp) < 0.5:
        print(f"!! WARNING: {_c} RH has almost no year-to-year spread (median sd {np.nanmedian(_sp):.2f} % points over {_full[0]}-{_full[-1]}): "
              f"it looks like a repeated climatology. Rebuild the RH input (Notebook A) before trusting this training.")
    else:
        print(f"RH year-to-year spread OK for {_c} (median sd {np.nanmedian(_sp):.1f} % points)")

## 2. Climatology, anomalies, scaling
Fitted on training months only, so nothing from validation or test leaks in. The final model uses all months.

In [ ]:
def shrink_clim(s, n, g, k_shrink, smooth, axis=1):
    c = (s + k_shrink * g) / (n + k_shrink)
    if smooth:
        c = 0.25 * np.roll(c, 1, axis) + 0.5 * c + 0.25 * np.roll(c, -1, axis)
    return c.astype("float32")

def fit_climatology(V, U, CAL, CID, elev, mm, n_c, n_bins, k_shrink, smooth):
    N, M, C = V.shape
    S_ = np.zeros((N, 12, C), "float32"); C_ = np.zeros((N, 12, C), "float32")
    for cm in range(12):
        sel = np.flatnonzero((CAL == cm) & mm)
        if len(sel) == 0: continue
        u = U[:, sel, :]
        S_[:, cm, :] = np.where(u, V[:, sel, :], 0).sum(1)
        C_[:, cm, :] = u.sum(1)
    edges, bins = [], np.zeros(N, int)
    for c in range(n_c):
        ic = np.flatnonzero(CID == c)
        e = np.quantile(elev[ic], np.linspace(0, 1, n_bins + 1)[1:-1]); edges.append(e); bins[ic] = np.digitize(elev[ic], e)
    all_mean = S_.sum(0, dtype="float64") / np.maximum(C_.sum(0, dtype="float64"), 1)
    G = np.zeros((n_c, n_bins, 12, C), "float32")
    for c in range(n_c):
        ic = CID == c
        cn = C_[ic].sum(0, dtype="float64")
        cmean = np.where(cn > 0, S_[ic].sum(0, dtype="float64") / np.maximum(cn, 1), all_mean)
        for b in range(n_bins):
            ib = ic & (bins == b)
            nb_, sb = C_[ib].sum(0, dtype="float64"), S_[ib].sum(0, dtype="float64")
            G[c, b] = np.where(nb_ > 0, sb / np.maximum(nb_, 1), cmean)
    clim = shrink_clim(S_, C_, G[CID, bins], k_shrink, smooth)
    return clim, G, edges, bins

def prepare(clim_end):
    ce = MIDX[clim_end]
    mm = np.arange(M) <= ce
    elev = ST[:, STN.index("elev")]
    clim, G, edges, bins = fit_climatology(VI, USABLE, CAL, CID, elev, mm, len(COUNTIES), ELEV_BINS, K_SHRINK, SMOOTH_CLIM)
    anom = np.where(USABLE, VI - clim[:, CAL, :], 0.0).astype("float32")
    an_sd = np.array([anom[:, mm, k][USABLE[:, mm, k]].std() for k in range(K)], "float32")
    anom_z = np.clip(anom / an_sd, -Z_CLIP, Z_CLIP).astype("float32"); del anom
    qual = np.where(CLEAN, 1.0, np.where(USABLE, 0.5, 0.0)).astype("float32")
    wxt = WX.astype("float32").copy(); ri = WXN.index("rain")
    wxt[..., ri] = np.log1p(np.clip(wxt[..., ri], 0, None))
    wx_ok = np.isfinite(wxt).all(-1)
    wx_mu = np.nanmean(wxt[:, mm, :], axis=(0, 1)); wx_sd = np.nanstd(wxt[:, mm, :], axis=(0, 1)) + 1e-6
    wxz = np.nan_to_num((wxt - wx_mu) / wx_sd).astype("float32")
    U4 = np.repeat(wx_ok[:, :, None], 4, 2)
    wclim, Gw, _, _ = fit_climatology(np.nan_to_num(wxt), U4, CAL, CID, elev, mm, len(COUNTIES), ELEV_BINS, K_SHRINK, False)
    wanom = np.where(U4, wxt - wclim[:, CAL, :], 0.0)
    wxa_sd = wanom[:, mm, :][wx_ok[:, mm]].std(axis=0) + 1e-6
    wxa = (wanom / wxa_sd).astype("float32"); del wanom
    wxcz = ((wclim - wx_mu) / wx_sd).astype("float32")
    st_mu, st_sd = ST.mean(0), ST.std(0) + 1e-6
    st_full = np.concatenate([(ST - st_mu) / st_sd, np.eye(len(COUNTIES))[CID]], 1).astype("float32")
    menc = np.stack([np.sin(2 * np.pi * CAL / 12), np.cos(2 * np.pi * CAL / 12)], 1).astype("float32")
    return dict(clim_end=clim_end, mm=mm, anom_z=anom_z, qual=qual, wxz=wxz, wxa=wxa, wxcz=wxcz, wx_ok=wx_ok, st_full=st_full, menc=menc,
                an_sd=an_sd, wx_mu=wx_mu, wx_sd=wx_sd, wxa_sd=wxa_sd, st_mu=st_mu, st_sd=st_sd, G=G, Gw=Gw, edges=edges)

# ---- windows ----
def build_valid(wx_ok, usable, L, H, pri_k, min_enc):
    N_, M_ = wx_ok.shape
    cs = lambda x: np.concatenate([np.zeros((N_, 1), np.int32), np.cumsum(x, 1, dtype=np.int32)], 1)
    cw, cp, ca = cs(wx_ok), cs(usable[:, :, pri_k]), cs(usable.any(2))
    t = np.arange(L, M_ - H + 1)
    ok = ((cw[:, t + H] - cw[:, t - L]) == (L + H)) & ((cp[:, t] - cp[:, t - L]) >= min_enc) & ((ca[:, t + H] - ca[:, t]) >= 1)
    return ok, t

def make_windows(valid, t, CID, H, train_end, val_end, holdout=None):
    te, ve = MIDX[train_end], MIDX[val_end]
    is_tr, is_va, is_te = (t + H - 1 <= te), (t > te) & (t + H - 1 <= ve), (t > ve)
    hold = COUNTIES.index(holdout) if holdout else None
    px_trv = np.ones(len(CID), bool) if hold is None else CID != hold
    px_te  = np.ones(len(CID), bool) if hold is None else CID == hold
    def collect(tm, pm):
        ii, jj = np.nonzero(valid & tm[None, :] & pm[:, None])
        return ii.astype(np.int32), t[jj].astype(np.int32)
    return dict(train=collect(is_tr, px_trv), val=collect(is_va, px_trv), test=collect(is_te, px_te),
                all=collect(np.ones(len(t), bool), np.ones(len(CID), bool)))

## 3. Model, batches and training

In [ ]:
class Seq2SeqLSTM(nn.Module):
    def __init__(self, n_enc, n_dec, n_static, n_out, hidden, layers, dropout, s_emb):
        super().__init__()
        self.static = nn.Sequential(nn.Linear(n_static, s_emb), nn.ReLU())
        p = dropout if layers > 1 else 0.0
        self.enc = nn.LSTM(n_enc + s_emb, hidden, layers, batch_first=True, dropout=p)
        self.dec = nn.LSTM(n_dec + s_emb, hidden, layers, batch_first=True, dropout=p)
        self.drop, self.head = nn.Dropout(dropout), nn.Linear(hidden, n_out)
    def forward(self, xe, xd, st):
        s = self.static(st)
        xe = torch.cat([xe, s[:, None, :].expand(-1, xe.size(1), -1)], -1)
        xd = torch.cat([xd, s[:, None, :].expand(-1, xd.size(1), -1)], -1)
        _, state = self.enc(xe)
        y, _ = self.dec(xd, state)
        return self.head(self.drop(y))

def to_torch(P):
    T = lambda a, dt=torch.float32: torch.as_tensor(a, dtype=dt, device=DEVICE)
    return dict(ANOM=T(P["anom_z"]), QUAL=T(P["qual"]), WXZ=T(P["wxz"]), WXA=T(P["wxa"]), WXCZ=T(P["wxcz"]), STZ=T(P["st_full"]),
                MENC=T(P["menc"]), CALT=T(CAL, torch.long), RAT=T(P["wx_sd"] / P["wxa_sd"]),
                AR_E=torch.arange(-L, 0, device=DEVICE), AR_D=torch.arange(0, H, device=DEVICE))

def batch(X, ii, tt, train=False, wx_mode="observed"):
    ii = torch.as_tensor(ii, dtype=torch.long, device=DEVICE); tt = torch.as_tensor(tt, dtype=torch.long, device=DEVICE)
    te, td, i2 = tt[:, None] + X["AR_E"], tt[:, None] + X["AR_D"], ii[:, None]
    xe = torch.cat([X["ANOM"][i2, te], X["QUAL"][i2, te], X["WXZ"][i2, te], X["WXA"][i2, te], X["MENC"][te]], -1)
    if wx_mode == "observed":
        raw, an = X["WXZ"][i2, td], X["WXA"][i2, td]
    else:
        raw = X["WXCZ"][i2, X["CALT"][td]]; an = torch.zeros_like(raw)
    if train:
        d = WX_NOISE * torch.randn_like(raw) + WX_BIAS_SD * torch.randn(raw.size(0), 1, raw.size(2), device=DEVICE)
        raw, an = raw + d, an + d * X["RAT"]
    xd = torch.cat([raw, an, X["MENC"][td]], -1)
    return xe, xd, X["STZ"][ii], X["ANOM"][i2, td], X["QUAL"][i2, td]

def weights_of(q):
    return (q >= 1).float() + QA2_WEIGHT * ((q > 0) & (q < 1)).float()

def wloss(pred, y, q):
    w = weights_of(q)
    return (w * F.huber_loss(pred, y, reduction="none", delta=HUBER_DELTA)).sum() / w.sum().clamp(min=1e-6)

def new_model(n_static):
    return Seq2SeqLSTM(2 * K + 8 + 2, 8 + 2, n_static, K, HIDDEN, LAYERS, DROPOUT, STATIC_EMB).to(DEVICE)

@torch.no_grad()
def val_loss(model, X, win, rng):
    ii, tt = win
    sel = rng.choice(len(ii), min(VAL_SAMPLES, len(ii)), replace=False)
    model.eval(); tot = wsum = 0.0
    for s in range(0, len(sel), 8192):
        xe, xd, st, y, q = batch(X, ii[sel[s:s+8192]], tt[sel[s:s+8192]])
        w = weights_of(q)
        l = (w * F.huber_loss(model(xe, xd, st), y, reduction="none", delta=HUBER_DELTA)).sum().item()
        tot += l; wsum += w.sum().item()
    return tot / max(wsum, 1e-6)

def train_one(X, win, n_static, seed, fixed_epochs=None, verbose=True):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = new_model(n_static)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=4) if fixed_epochs is None else None
    ii, tt = win["train"]
    best, best_state, best_ep, bad, hist = np.inf, None, 0, 0, []
    for ep in range(1, (fixed_epochs or MAX_EPOCHS) + 1):
        model.train(); pick = rng.integers(0, len(ii), EPOCH_SAMPLES); run = 0.0
        for b in range(0, EPOCH_SAMPLES, BATCH):
            sl = pick[b:b + BATCH]
            xe, xd, st, y, q = batch(X, ii[sl], tt[sl], train=True)
            loss = wloss(model(xe, xd, st), y, q)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM); opt.step(); run += loss.item() * len(sl)
        tr = run / EPOCH_SAMPLES
        if fixed_epochs is None:
            vl = val_loss(model, X, win["val"], np.random.default_rng(0)); sched.step(vl); hist.append((ep, tr, vl))
            if vl < best - 1e-4: best, best_state, best_ep, bad = vl, copy.deepcopy(model.state_dict()), ep, 0
            else: bad += 1
            if verbose and (ep % 5 == 0 or ep == 1): print(f"  seed {seed} epoch {ep:3d} train {tr:.4f} val {vl:.4f} (best {best:.4f} @ {best_ep})", flush=True)
            if bad >= PATIENCE: break
        elif verbose and ep % 10 == 0: print(f"  seed {seed} epoch {ep}/{fixed_epochs} train {tr:.4f}", flush=True)
    if best_state is not None: model.load_state_dict(best_state)
    return model, hist, (best_ep if fixed_epochs is None else fixed_epochs)

## 4. Evaluation
Scored on clear (QA 0) targets in standardised units. Baselines: **climatology** (anomaly 0) and **persistence** (last usable anomaly).

In [ ]:
@torch.no_grad()
def evaluate(X, P, models, ii, tt, wx_mode="observed", chunk=20000):
    nC = len(COUNTIES)
    keys = ("sse_m", "sse_c", "sse_p", "n")
    tot = {k: np.zeros((H, K)) for k in keys}
    byc = {k: np.zeros((nC, H, K)) for k in keys}
    bym = {k: np.zeros((12, H, K)) for k in keys}
    for m_ in models: m_.eval()
    for s in range(0, len(ii), chunk):
        a, b = ii[s:s+chunk], tt[s:s+chunk]
        xe, xd, st, y, q = batch(X, a, b, wx_mode=wx_mode)
        pred = torch.stack([m_(xe, xd, st) for m_ in models]).mean(0)
        m = (q >= 1).float()                                                  # clear targets only
        use = xe[:, :, K:2 * K] > 0                                           # usable history months (quality channel)
        rev = use.flip(1); has = rev.any(1)
        last = L - 1 - rev.float().argmax(1)                                  # index of the last usable month, per index
        pers = torch.gather(xe[:, :, :K], 1, last[:, None, :]).squeeze(1) * has
        pers = pers[:, None, :].expand(-1, H, -1)
        err = dict(sse_m=((pred - y) ** 2) * m, sse_c=(y ** 2) * m, sse_p=((pers - y) ** 2) * m, n=m)
        cb = torch.as_tensor(CID[a], device=DEVICE); mb = torch.as_tensor(CAL[b], device=DEVICE)
        for k in keys:
            e = err[k]; tot[k] += e.sum(0).cpu().numpy()
            for c in range(nC): byc[k][c] += e[cb == c].sum(0).cpu().numpy()
            for mo in range(12): bym[k][mo] += e[mb == mo].sum(0).cpu().numpy()
    return dict(tot=tot, byc=byc, bym=bym)

def skill(sse_m, sse_ref):
    with np.errstate(invalid="ignore", divide="ignore"):
        return 1 - sse_m / sse_ref

def skill_table(R, base="sse_c"):
    return pd.DataFrame(skill(R["tot"]["sse_m"], R["tot"][base]), index=[f"lead {h+1}" for h in range(H)], columns=VIS)

def summarize(R, title=""):
    sc, sp = skill_table(R, "sse_c"), skill_table(R, "sse_p")
    out = pd.DataFrame({"skill vs climatology": sc.mean(1), "skill vs persistence": sp.mean(1),
                        "share of indices > 0 (vs clim)": (sc > 0).mean(1), "n targets": R["tot"]["n"].sum(1)})
    print(title); print(out.round(3).to_string()); return sc, sp

def plot_skill(sc, sp, title):
    fig, ax = plt.subplots(1, 2, figsize=(13, max(3.8, 0.3 * K)), gridspec_kw={"width_ratios": [1.5, 1]})
    im = ax[0].imshow(sc.values.T, cmap="RdYlGn", vmin=-0.5, vmax=0.5, aspect="auto")
    ax[0].set_xticks(range(H)); ax[0].set_xticklabels(range(1, H + 1)); ax[0].set_yticks(range(K)); ax[0].set_yticklabels(VIS)
    ax[0].set_xlabel("lead (months ahead)"); ax[0].set_title(title + ": skill vs climatology"); plt.colorbar(im, ax=ax[0])
    for i in range(K):
        for j in range(H): ax[0].text(j, i, f"{sc.values[j, i]:.2f}", ha="center", va="center", fontsize=7)
    ax[1].plot(range(1, H + 1), sc.mean(1), "o-", label="vs climatology"); ax[1].plot(range(1, H + 1), sp.mean(1), "s-", label="vs persistence")
    ax[1].axhline(0, color="k", lw=.7); ax[1].set_xlabel("lead (months ahead)"); ax[1].set_ylabel("mean skill over indices"); ax[1].legend()
    plt.tight_layout(); plt.show()

## 5. Train, validate, test
Trains `N_SEEDS` models with early stopping on the validation year and scores the test period. Window counts should be near 30 / 6 / 15 per pixel (train / val / test).

In [ ]:
t0 = time.time()
P = prepare(TRAIN_END)
X = to_torch(P)
valid, tgrid = build_valid(P["wx_ok"], USABLE, L, H, PRI_K, MIN_ENC_USABLE)
WIN = make_windows(valid, tgrid, CID, H, TRAIN_END, VAL_END, HOLDOUT_COUNTY)
print({k: len(v[0]) for k, v in WIN.items()}, "| per pixel (train/val/test):", [round(len(WIN[k][0]) / N, 1) for k in ("train", "val", "test")])
n_static = P["st_full"].shape[1]
print(f"prepared in {time.time()-t0:.0f}s | anomaly sd per index:", dict(zip(VIS, P["an_sd"].round(3))))

print("decoder-weather offset used in training (WX_BIAS_SD, 1 SD, real units):",
      {_v: round(WX_BIAS_SD * float(P["wx_sd"][WXN.index(_v)]), 2) for _v in ("temp", "rh", "wind")})
MODELS, HISTS, BEST_EP = [], [], []
for s in range(N_SEEDS):
    print(f"--- training seed {SEED + s} ---", flush=True)
    mdl, hist, bep = train_one(X, WIN, n_static, SEED + s)
    MODELS.append(mdl); HISTS.append(hist); BEST_EP.append(bep)
    print(f"  best epoch {bep}, {time.time()-t0:.0f}s elapsed")

fig, ax = plt.subplots(figsize=(6, 3.2))
for i, h in enumerate(HISTS):
    h = np.array(h); ax.plot(h[:, 0], h[:, 2], label=f"val seed {i}"); ax.plot(h[:, 0], h[:, 1], "--", alpha=.4)
ax.set_xlabel("epoch"); ax.set_ylabel("weighted Huber loss"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
ii_te, tt_te = WIN["test"]
R_obs = evaluate(X, P, MODELS, ii_te, tt_te, "observed")
sc, sp = summarize(R_obs, "\nTEST with OBSERVED weather in the forecast months (upper bound):")
plot_skill(sc, sp, "test, observed weather")
print("\nskill vs climatology by index and lead:"); print(sc.round(2).to_string())

### 5b. How much comes from the weather?
Same test, with normal weather in the forecast months. The gap is the skill that depends on knowing the coming weather.

In [ ]:
R_clim = evaluate(X, P, MODELS, ii_te, tt_te, "climatology")
sc_c, sp_c = summarize(R_clim, "TEST with NORMAL (climatological) weather in the forecast months (no weather information):")
gain = (sc.mean(1) - sc_c.mean(1)).rename("skill gained from knowing the weather")
print("\n", gain.round(3).to_string())

byc = pd.DataFrame({c: skill(R_obs["byc"]["sse_m"][i], R_obs["byc"]["sse_c"][i]).mean(1) for i, c in enumerate(COUNTIES)}, index=[f"lead {h+1}" for h in range(H)])
print("\nmean skill vs climatology by county:"); print(byc.round(3).to_string())
bym = skill(R_obs["bym"]["sse_m"].sum(1), R_obs["bym"]["sse_c"].sum(1))                 # [12, K]: pooled over leads
print("\nmean skill by calendar month of the first target month (pooled over leads):")
print(pd.Series(np.nanmean(bym, 1), index=[f"{m+1:02d}" for m in range(12)]).round(2).to_string())

## 6. Optional: rolling-origin evaluation
Retrains at three origins (set `RUN_ROLLING_ORIGIN = True`) to show how stable the skill is.

In [ ]:
RUN_ROLLING_ORIGIN = False
ROLLING = None
if RUN_ROLLING_ORIGIN:
    rows = []
    for te_, ve_ in [("2022-12", "2023-12"), ("2023-12", "2024-12"), ("2024-12", "2025-12")]:
        print(f"\n=== origin: train to {te_}, validate {ve_}, test after ===", flush=True)
        P_r = prepare(te_); X_r = to_torch(P_r)
        v_r, t_r = build_valid(P_r["wx_ok"], USABLE, L, H, PRI_K, MIN_ENC_USABLE)
        W_r = make_windows(v_r, t_r, CID, H, te_, ve_, HOLDOUT_COUNTY)
        ms = [train_one(X_r, W_r, P_r["st_full"].shape[1], SEED + s, verbose=False)[0] for s in range(ROLLING_SEEDS)]
        R_r = evaluate(X_r, P_r, ms, *W_r["test"])
        sk = skill_table(R_r).mean(1)
        rows.append(pd.Series(sk.values, index=sk.index, name=f"train to {te_}"))
        del X_r, P_r, ms; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    ROLLING = pd.DataFrame(rows).T
    ROLLING["mean over origins"] = ROLLING.mean(1)
    print("\nmean skill vs climatology by lead and origin:"); print(ROLLING.round(3).to_string())
else:
    print("skipped (RUN_ROLLING_ORIGIN = False)")

## 7. Final model
Refits on all windows and saves the model with the scalers and climatology that Notebook C needs.

In [ ]:
if FINAL_FIT:
    LAST = MONTHS[-1]
    P_f = prepare(LAST); X_f = to_torch(P_f)
    v_f, t_f = build_valid(P_f["wx_ok"], USABLE, L, H, PRI_K, MIN_ENC_USABLE)
    W_f = make_windows(v_f, t_f, CID, H, TRAIN_END, VAL_END, None)
    W_f["train"] = W_f["all"]
    ep_fixed = int(round(np.mean(BEST_EP) * 1.15)) + 1
    print(f"final fit on {len(W_f['train'][0])} windows, {ep_fixed} epochs, {N_SEEDS} seeds", flush=True)
    final_models = [train_one(X_f, W_f, P_f["st_full"].shape[1], SEED + 100 + s, fixed_epochs=ep_fixed)[0] for s in range(N_SEEDS)]
    cfg = dict(L=L, H=H, K=K, VIS=VIS, WXN=WXN, STN=STN, COUNTIES=COUNTIES, MONTHS=MONTHS, PRIMARY=PRIMARY, MIN_ENC_USABLE=MIN_ENC_USABLE,
               HIDDEN=HIDDEN, LAYERS=LAYERS, STATIC_EMB=STATIC_EMB, DROPOUT=DROPOUT, n_static=P_f["st_full"].shape[1],
               K_SHRINK=K_SHRINK, ELEV_BINS=ELEV_BINS, SMOOTH_CLIM=SMOOTH_CLIM, Z_CLIP=Z_CLIP, QA2_WEIGHT=QA2_WEIGHT, QA_SOURCE=QA_SOURCE,
               clim_end=LAST, epochs=ep_fixed, trained=time.strftime("%Y-%m-%d %H:%M"))
    norm = {k: P_f[k] for k in ("an_sd", "wx_mu", "wx_sd", "wxa_sd", "st_mu", "st_sd", "G", "Gw")}
    norm["edges"] = [np.asarray(e) for e in P_f["edges"]]
    art = dict(cfg=cfg, norm=norm, states=[m.state_dict() for m in final_models])
    torch.save(art, os.path.join(MODEL_DIR, "final_model.pt"))
    print("saved", os.path.join(MODEL_DIR, "final_model.pt"))
    del X_f
sc.to_csv(os.path.join(MODEL_DIR, "test_skill_observed_weather.csv")); sc_c.to_csv(os.path.join(MODEL_DIR, "test_skill_climatological_weather.csv"))
if ROLLING is not None: ROLLING.to_csv(os.path.join(MODEL_DIR, "rolling_origin_skill.csv"))

## Checks before moving on
1. Window counts near 30 / 6 / 15 per pixel.
2. Validation loss flattening while training loss falls = over-fitting: raise `DROPOUT` or `WEIGHT_DECAY`.
3. Lead-1 skill clearly above 0, falling with lead.
4. The weather-gain table shows how much SEAS5 quality matters.
5. Indices with negative skill at all leads are not forecastable: report or drop them.
6. In Notebook C, bias-correct SEAS5 with D's table before it enters the decoder.